# 第十階段:擴充到多場 —— 跨比賽的 play 瀏覽器

**最終目標**:做一個「傳球空間分析」工具,給教練或球探使用。

**本階段只做一件事**:把第九階段**只能看一場**的互動工具,擴充成**能看全部 122 場**。從「選 play」升級成「**選比賽 → 選 play → 出卡**」兩層選單。

---

### 從一場到多場,關鍵改變在哪?

前面九個階段都寫死用 `SAMPLE_GAME_ID` 這一場。但整份資料集有 **122 場**(2021 球季第 1–8 週),tracking 檔共約 850MB。

- **不能一次全載**:850MB 全讀進記憶體不切實際,也沒必要。
- **做法:選到才讀 + 快取**。使用者選了某場,才去讀那場的 `tracking_<gameId>.csv`;讀過的場存在記憶體,下次再選同一場就不重讀。
- **`plays` / `players` / `pff` 是全季共用的**,讀一次就好(它們不大)。

> 代價:切換到沒讀過的比賽,要讀一個新檔(幾 MB,約一兩秒)。用快取後,回頭看讀過的場是即時的。

## 步驟 0:載入工具

- **輸入**:無
- **輸出**:工具 + `ipywidgets`
- **目的**:跟第九階段一樣,多了跨場需要的東西。

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from scipy.spatial import cKDTree
import ipywidgets as widgets
from IPython.display import display

plt.rcParams["font.sans-serif"] = ["PingFang TC", "Heiti TC", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False

sys.path.append(str(Path.cwd().parent))
from config import DATA_DIR

print("ipywidgets", widgets.__version__)

ipywidgets 8.1.9


## 步驟 1:載入全季共用資料 + 建立 tracking 快取讀取器

- **輸入**:`games`、`plays`、`players`、`pff`(全季,各讀一次)
- **輸出**:四個 DataFrame + `get_tracking(game_id)` 函式
- **目的**:這是本階段的核心工程。`get_tracking` 負責「選到才讀 + 快取」:第一次讀某場會從硬碟載入,之後存進 `_tracking_cache` 重複使用。

> `games` 用來列出 122 場比賽;`plays`/`players`/`pff` 全季共用。只有 tracking 是逐場載入。

In [2]:
games = pd.read_csv(DATA_DIR / "games.csv")
plays = pd.read_csv(DATA_DIR / "plays.csv")
players = pd.read_csv(DATA_DIR / "players.csv")
pff = pd.read_csv(DATA_DIR / "pffScoutingData.csv")

_tracking_cache = {}  # gameId -> tracking DataFrame

def get_tracking(game_id):
    """選到才讀 + 快取。第一次讀硬碟,之後直接回傳記憶體裡的。"""
    if game_id not in _tracking_cache:
        _tracking_cache[game_id] = pd.read_csv(DATA_DIR / "tracking" / f"tracking_{game_id}.csv")
    return _tracking_cache[game_id]


print("全季資料:", len(games), "場比賽 /", len(plays), "個 play /", len(players), "位球員")
print("tracking 採『選到才讀 + 快取』,目前快取:", len(_tracking_cache), "場")

全季資料: 122 場比賽 / 8557 個 play / 1679 位球員
tracking 採『選到才讀 + 快取』,目前快取: 0 場


## 步驟 2:play 卡繪圖邏輯(改成吃任一場的 tracking)

- **輸入**:`game_id`、`play_id`
- **輸出**:`prepare_play(game_id, play_id)`、panel 函式、`play_card(game_id, play_id)`
- **目的**:沿用第七/九階段的 play 卡,但把寫死的單場 tracking 換成 `get_tracking(game_id)`,並讓 `prepare_play` 多吃一個 `game_id`。

> 這是這一階段唯一需要動 play 卡程式的地方:把「場」從全域常數變成參數。

In [3]:
FIELD_X, FIELD_Y, STEP = 120.0, 53.3, 0.5
_gx = np.arange(0, FIELD_X, STEP) + STEP / 2
_gy = np.arange(0, FIELD_Y, STEP) + STEP / 2
_GX, _GY = np.meshgrid(_gx, _gy)
GRID = np.column_stack([_GX.ravel(), _GY.ravel()])


def prepare_play(game_id, play_id):
    pl = plays[(plays["gameId"] == game_id) & (plays["playId"] == play_id)]
    if pl.empty:
        return None
    pl = pl.iloc[0]
    tracking = get_tracking(game_id)
    pt = tracking[(tracking["gameId"] == game_id) & (tracking["playId"] == play_id)]
    sr = pt.loc[pt["event"] == "ball_snap", "frameId"]
    pr = pt.loc[pt["event"] == "pass_forward", "frameId"]
    if sr.empty or pr.empty:
        return None
    snap_frame, pass_frame = sr.iloc[0], pr.iloc[0]
    roles = pff[(pff["gameId"] == game_id) & (pff["playId"] == play_id)][["nflId", "pff_role"]]
    seg = (
        pt[(pt["frameId"] >= snap_frame) & (pt["frameId"] <= pass_frame)]
        .merge(players[["nflId", "displayName"]], on="nflId", how="left")
        .merge(roles, on="nflId", how="left")
    )
    return {"game_id": game_id, "play_id": play_id, "info": pl,
            "off_team": pl["possessionTeam"], "def_team": pl["defensiveTeam"],
            "snap_frame": snap_frame, "pass_frame": pass_frame, "seg": seg}


def _p2s(px, py, ax0, ay0, bx, by):
    vx, vy = bx - ax0, by - ay0
    wx, wy = px - ax0, py - ay0
    L2 = vx * vx + vy * vy
    if L2 == 0:
        return np.hypot(px - ax0, py - ay0), 0.0
    t = (wx * vx + wy * vy) / L2
    tc = max(0.0, min(1.0, t))
    return np.hypot(px - (ax0 + tc * vx), py - (ay0 + tc * vy)), t


def _snapshot(d, ax):
    s = d["seg"][(d["seg"]["frameId"] == d["snap_frame"]) & d["seg"]["nflId"].notna()]
    o = s[s["team"] == d["off_team"]]; f = s[s["team"] == d["def_team"]]
    ax.scatter(o["x"], o["y"], c="tab:blue", s=60, label=f"進攻 {d['off_team']}")
    ax.scatter(f["x"], f["y"], c="tab:red", s=60, label=f"防守 {d['def_team']}")
    ax.set_xlim(0, FIELD_X); ax.set_ylim(0, FIELD_Y); ax.set_aspect("equal")
    ax.set_title("① 發球當下場上位置"); ax.legend(loc="upper right", fontsize=7)


def _voronoi(d, ax):
    s = d["seg"][(d["seg"]["frameId"] == d["snap_frame"]) & d["seg"]["nflId"].notna()]
    _, idx = cKDTree(s[["x", "y"]].to_numpy()).query(GRID)
    owner = s["team"].to_numpy()[idx]
    off_pct = (owner == d["off_team"]).sum() / len(owner) * 100
    ax.imshow((owner == d["def_team"]).astype(int).reshape(_GX.shape), origin="lower",
              extent=[0, FIELD_X, 0, FIELD_Y], cmap=ListedColormap(["#a6c8ff", "#ffb3b3"]),
              alpha=0.7, aspect="equal")
    ax.scatter(s[s["team"] == d["off_team"]]["x"], s[s["team"] == d["off_team"]]["y"], c="tab:blue", s=40)
    ax.scatter(s[s["team"] == d["def_team"]]["x"], s[s["team"] == d["def_team"]]["y"], c="tab:red", s=40)
    ax.set_xlim(0, FIELD_X); ax.set_ylim(0, FIELD_Y)
    ax.set_title(f"② Voronoi 控制區(進攻控制 {off_pct:.0f}%)")


def _separation(d, ax):
    rec = d["seg"][d["seg"]["pff_role"] == "Pass Route"]
    dfn = d["seg"][d["seg"]["pff_role"] == "Coverage"]
    for name, rg in rec.groupby("displayName"):
        xs, ys = [], []
        for fid, rrow in rg.groupby("frameId"):
            dd = dfn[dfn["frameId"] == fid][["x", "y"]].to_numpy()
            r0 = rrow.iloc[0]
            if len(dd):
                xs.append((fid - d["snap_frame"]) * 0.1)
                ys.append(np.sqrt((dd[:, 0] - r0["x"]) ** 2 + (dd[:, 1] - r0["y"]) ** 2).min())
        ax.plot(xs, ys, marker="o", markersize=2, label=name)
    ax.axvline((d["pass_frame"] - d["snap_frame"]) * 0.1, color="gray", linestyle="--", linewidth=1)
    ax.set_xlabel("發球後秒數"); ax.set_ylabel("separation(碼)")
    ax.set_title("③ 接球員與最近防守者距離"); ax.legend(fontsize=6, loc="upper left"); ax.grid(True, alpha=0.3)


def _lanes(d, ax):
    fr = d["seg"][(d["seg"]["frameId"] == d["pass_frame"]) & d["seg"]["nflId"].notna()]
    qb = fr[fr["pff_role"] == "Pass"]
    if qb.empty:
        ax.set_title("④ 傳球路徑(此 play 無 QB 標記)"); return
    qb = qb.iloc[0]
    rec = fr[fr["pff_role"] == "Pass Route"]; dfn = fr[fr["pff_role"] == "Coverage"]
    for _, r in rec.iterrows():
        blk = [dist for _, dd in dfn.iterrows()
               for dist, t in [_p2s(dd["x"], dd["y"], qb["x"], qb["y"], r["x"], r["y"])] if 0 <= t <= 1]
        md = min(blk) if blk else np.nan
        color = "tab:green" if (np.isnan(md) or md >= 3) else ("orange" if md >= 1.5 else "tab:red")
        ax.plot([qb["x"], r["x"]], [qb["y"], r["y"]], "--", color=color, linewidth=1.3)
    ax.scatter(rec["x"], rec["y"], c="tab:blue", s=50, label="接球員")
    ax.scatter(dfn["x"], dfn["y"], c="tab:red", s=50, label="防守員")
    ax.scatter([qb["x"]], [qb["y"]], c="black", s=110, marker="*", label="QB")
    ax.set_xlim(0, FIELD_X); ax.set_ylim(0, FIELD_Y); ax.set_aspect("equal")
    ax.set_title("④ 傳球路徑(僅供參考:平面指標,對深傳不準)", color="dimgray")
    ax.legend(fontsize=7, loc="upper right")


def play_card(game_id, play_id):
    d = prepare_play(game_id, play_id)
    if d is None:
        print(f"gameId={game_id} playId={play_id} 不是傳球 play,無法製卡。"); return
    info = d["info"]
    fig, axes = plt.subplots(2, 2, figsize=(15, 9))
    _snapshot(d, axes[0, 0]); _voronoi(d, axes[0, 1])
    _separation(d, axes[1, 0]); _lanes(d, axes[1, 1])
    title = (f"Play 卡 — game {game_id} / play {play_id}  |  {d['off_team']}(攻) vs {d['def_team']}(守)"
             f"  |  第 {info.get('down','?')} 檔還差 {info.get('yardsToGo','?')} 碼  |  結果 {info.get('passResult','?')}")
    fig.suptitle(title, fontsize=13, y=0.99)
    fig.text(0.5, 0.005, str(info.get("playDescription", "")), ha="center", fontsize=9, color="dimgray")
    fig.tight_layout(rect=[0, 0.02, 1, 0.97])
    plt.show()


print("play 卡邏輯已就緒(可吃任一場)。")

play 卡邏輯已就緒(可吃任一場)。


## 步驟 3:列出傳球 play 的小工具

- **輸入**:`game_id`
- **輸出**:那場所有傳球 play 的下拉選項
- **目的**:選單需要「某一場有哪些傳球 play」。這一步會讀該場 tracking(透過快取),找出有 snap + pass_forward 的 play,組成選項。

In [4]:
def pass_play_options(game_id):
    tracking = get_tracking(game_id)
    opts = []
    for pid in sorted(tracking["playId"].unique()):
        pt = tracking[tracking["playId"] == pid]
        if not ((pt["event"] == "ball_snap").any() and (pt["event"] == "pass_forward").any()):
            continue
        row = plays[(plays["gameId"] == game_id) & (plays["playId"] == pid)]
        if row.empty:
            continue
        row = row.iloc[0]
        desc = str(row.get("playDescription", ""))
        label = f"#{pid} | 第{row.get('down','?')}檔 | {desc[:48]}"
        opts.append((label, pid))
    return opts


# 測一場
demo_game = games["gameId"].iloc[0]
demo_opts = pass_play_options(demo_game)
print(f"gameId {demo_game} 的傳球 play 數:", len(demo_opts))
print("範例:", demo_opts[0][0])

gameId 2021090900 的傳球 play 數: 92
範例: #97 | 第3檔 | (13:33) (Shotgun) T.Brady pass incomplete deep r


## 步驟 4:兩層選單 —— 選比賽 → 選 play → 出卡

- **輸入**:`games`、`play_card()`
- **輸出**:兩個連動的下拉選單 + play 卡
- **目的**:這是本階段的成品。選了比賽,第二個選單自動更新成那場的傳球 play;選了 play,就出卡。

> **在 Jupyter 執行**才會看到選單。選新的比賽時,若是第一次看那場會讀檔(約一兩秒),之後因為有快取就很快。

In [5]:
# 比賽選單:顯示 週次 + 主客隊,值是 gameId
game_options = [
    (f"第{r['week']}週 | {r['visitorTeamAbbr']} @ {r['homeTeamAbbr']} (#{r['gameId']})", r["gameId"])
    for _, r in games.sort_values(["week", "gameId"]).iterrows()
]

game_dd = widgets.Dropdown(options=game_options, description="選比賽:",
                           layout=widgets.Layout(width="60%"), style={"description_width": "initial"})
play_dd = widgets.Dropdown(options=pass_play_options(game_dd.value), description="選 play:",
                           layout=widgets.Layout(width="90%"), style={"description_width": "initial"})
out = widgets.Output()

def _render():
    with out:
        out.clear_output(wait=True)
        play_card(game_dd.value, play_dd.value)

def _on_game_change(change):
    if change["type"] == "change" and change["name"] == "value":
        # 換比賽 -> 更新 play 選單(讀新場,有快取就快)
        play_dd.options = pass_play_options(change["new"])
        if play_dd.options:
            play_dd.value = play_dd.options[0][1]
        _render()

def _on_play_change(change):
    if change["type"] == "change" and change["name"] == "value":
        _render()

game_dd.observe(_on_game_change)
play_dd.observe(_on_play_change)

# 先畫一次預設(靜態瀏覽時也看得到)
_render()
display(game_dd, play_dd, out)

Dropdown(description='選比賽:', layout=Layout(width='60%'), options=(('第1週 | DAL @ TB (#2021090900)', 2021090900)…

Dropdown(description='選 play:', layout=Layout(width='90%'), options=(('#97 | 第3檔 | (13:33) (Shotgun) T.Brady p…

Output()

## 本階段完成了什麼

- 把工具從**只能看一場**擴充成**能看全部 122 場**,升級為「選比賽 → 選 play → 出卡」兩層選單。
- 用 **「選到才讀 + 快取」** 處理 850MB 的 tracking:不一次全載,選哪場讀哪場,讀過的留在記憶體。
- `plays` / `players` / `pff` 全季共用、只讀一次;play 卡邏輯沿用前面階段,只把「場」從常數改成參數。
- 兩個選單連動:換比賽會自動更新該場的 play 清單。

**意義**:工具終於能分析**整個球季**,而不是一場 demo。這也讓第四階段的統計(平均 separation 等)有機會累積成跨場、更有份量的數據。

**下一階段(等你確認理解後再開始)**:
1. **跨場統計**:沿用第四階段方法,但掃過多場,算整個球季的 separation 排行(逐場讀、累積結果,注意時間)。
2. **批次匯出報告**:挑定幾場,一次把 play 卡存成圖檔 / PDF。
3. **打包成獨立 app**(Streamlit),不需開 Jupyter。